<a href="https://colab.research.google.com/github/tarunpahade18/ml-notebooks/blob/main/lung_cancer_cnn_fixed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lung histopathology CNN (LC25000, 3 classes)
Research/education only. Not a medical device and not for diagnosis.

Fixes the old notebook: the 256x256 model that overwrote the 128x128 CNN caused a shape ValueError. Data now comes from a public Hugging Face copy of LC25000 (no Kaggle token).

In [1]:
!pip -q install datasets

In [2]:
import numpy as np, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
IMG_SIZE=128; EPOCHS=10; BATCH_SIZE=64; SEED=2022
ds = load_dataset('1aurent/LC25000', split='train')
ds = ds.filter(lambda r: r['organ']==0)  # lung only
names = ['lung_benign','lung_adenocarcinoma','lung_squamous_cell_carcinoma']
print(len(ds), np.bincount(ds['label']))

README.md:   0%|          | 0.00/1.71k [00:00<?, ?B/s]

data/train-00000-of-00004-40f80292828d4b(…): reconstructing file:   0%|          |  0.00B /  387MB            

data/train-00000-of-00004-40f80292828d4b(…): downloading bytes:           |  0.00B            

data/train-00001-of-00004-de395cdb9595cd(…): reconstructing file:   0%|          |  0.00B /  375MB            

data/train-00001-of-00004-de395cdb9595cd(…): downloading bytes:           |  0.00B            

data/train-00002-of-00004-8b746b230324f3(…): reconstructing file:   0%|          |  0.00B /  533MB            

... (2 outputs left out)

data/train-00003-of-00004-402d3c6cb6e672(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/25000 [00:00<?, ? examples/s]

15000 [5000 5000 5000]


In [3]:
X = np.stack([np.asarray(im.convert('RGB').resize((IMG_SIZE,IMG_SIZE))) for im in ds['image']]).astype('uint8')
Y = np.array(ds['label'])
X_train, X_tmp, y_train, y_tmp = train_test_split(X, Y, test_size=0.3, stratify=Y, random_state=SEED)
X_val, X_test, y_val, y_test = train_test_split(X_tmp, y_tmp, test_size=0.5, stratify=y_tmp, random_state=SEED)
print(X_train.shape, X_val.shape, X_test.shape)

(10500, 128, 128, 3) (2250, 128, 128, 3) (2250, 128, 128, 3)


In [4]:
model = keras.Sequential([
    layers.Input((IMG_SIZE,IMG_SIZE,3)),
    layers.Rescaling(1/255.),
    layers.Conv2D(32,3,activation='relu'), layers.MaxPooling2D(),
    layers.Conv2D(64,3,activation='relu'), layers.MaxPooling2D(),
    layers.Conv2D(128,3,activation='relu'), layers.MaxPooling2D(),
    layers.Flatten(), layers.Dense(128,activation='relu'), layers.Dropout(0.3),
    layers.Dense(3,activation='softmax')])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
cb=[keras.callbacks.EarlyStopping(patience=3,monitor='val_accuracy',restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss',patience=2,factor=0.5)]
history = model.fit(X_train,y_train,validation_data=(X_val,y_val),batch_size=BATCH_SIZE,epochs=EPOCHS,callbacks=cb)

Epoch 1/10
165/165 ━━━━━━━━━━━━━━━━━━━━ 29s 104ms/step - accuracy: 0.8029 - loss: 0.4422 - val_accuracy: 0.8702 - val_loss: 0.2951 - learning_rate: 0.0010
Epoch 2/10
165/165 ━━━━━━━━━━━━━━━━━━━━ 19s 33ms/step - accuracy: 0.8927 - loss: 0.2653 - val_accuracy: 0.9262 - val_loss: 0.1964 - learning_rate: 0.0010
Epoch 3/10
165/165 ━━━━━━━━━━━━━━━━━━━━ 5s 32ms/step - accuracy: 0.9144 - loss: 0.2160 - val_accuracy: 0.9213 - val_loss: 0.1931 - learning_rate: 0.0010
Epoch 4/10
165/165 ━━━━━━━━━━━━━━━━━━━━ 6s 35ms/step - accuracy: 0.9272 - loss: 0.1741 - val_accuracy: 0.9244 - val_loss: 0.1787 - learning_rate: 0.0010
Epoch 5/10
165/165 ━━━━━━━━━━━━━━━━━━━━ 5s 32ms/step - accuracy: 0.9450 - loss: 0.1405 - val_accuracy: 0.9324 - val_loss: 0.1689 - learning_rate: 0.0010
Epoch 6/10
165/165 ━━━━━━━━━━━━━━━━━━━━ 5s 32ms/step - accuracy: 0.9451 - loss: 0.1405 - val_accuracy: 0.9489 - val_loss: 0.1349 - learning_rate: 0.0010
Epoch 7/10
165/165 ━━━━━━━━━━━━━━━━━━━━ 5s 33ms/step - accuracy: 0.9591 - loss:

In [5]:
loss, acc = model.evaluate(X_test,y_test,verbose=0)
print('held-out test accuracy', acc)
pred = model.predict(X_test).argmax(1)
print(classification_report(y_test,pred,target_names=names))
print(confusion_matrix(y_test,pred))
# CAVEAT: LC25000 images are augmentations of ~750 originals, so a random split can put near-copies of an image in train and test. Accuracy is likely optimistic.

held-out test accuracy 0.964888870716095
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step
                              precision    recall  f1-score   support

                 lung_benign       1.00      1.00      1.00       750
         lung_adenocarcinoma       0.96      0.93      0.95       750
lung_squamous_cell_carcinoma       0.94      0.97      0.95       750

                    accuracy                           0.96      2250
                   macro avg       0.97      0.96      0.96      2250
                weighted avg       0.97      0.96      0.96      2250

[[747   3   0]
 [  0 700  50]
 [  0  26 724]]


In [6]:
model.save('lung_cancer_cnn.keras')
from google.colab import files  # optional download